# ML-01 Stage 0 Feature Build

목적: `00_ml_01_fb_author_contract.ipynb`에서 사용자가 작성한 최종 `fb_input_feature_contract.csv`를 읽고 feature build만 실행한다.

이 노트북은 contract를 작성하거나 `ml_inputs` 승인 배치를 수행하지 않는다.

1. 코드 전체 요약
    - 이 노트북은 ML-01 Stage 0 feature build 실행용 노트북이다.  
    - 이전 단계에서 작성된 fb_input_feature_contract.csv를 읽고 검증한 뒤, contract 기준으로 생성 대상 feature와 encoding 대상 feature를 확정한다. 
    - 이후 원본 parquet에서 feature를 생성하고, encoding 결과와 feature info를 fb_outputs 경로에 저장한다.
2. 데이터 흐름 요약
    ```
    SOURCE_PARQUET_PATH
        -> build_features()
        -> build_result.feature_frame
        -> encode_split_frame()
        -> Xy train/val/test 후보 산출물, output contract, encoding manifest

    FB_INPUT_CONTRACT_PATH
        -> validate_fb_input_contract()
        -> contract_df
        -> BUILD_FEATURE_SPECS / ENCODING_SPECS

    BUILD_FEATURE_SPECS
        -> FeatureBuildConfig
        -> build_features()

    ENCODING_SPECS
        -> encode_split_frame()
        -> categorical encoding 및 split frame 저장
        
    build_result.feature_info
        -> feature_info.csv 저장
    ```
3. 변경 시 주의점
    - EXPORT_EXPERIMENT_ID, RUN_ID, ARTIFACT_PREFIX를 바꾸면 입력 contract와 출력 artifact 경로가 모두 바뀐다.
    - SOURCE_PARQUET_PATH를 바꾸면 feature build 대상 원본 데이터가 바뀐다.
    - OVERWRITE_FB_OUTPUTS=False이면 기존 산출물이 있을 때 덮어쓰지 않고 실패한다.
    - SAMPLE_ROWS=None은 전체 데이터 실행이다. 디버깅 시 정수로 줄일 수 있지만, sample 결과는 성능 판단에 사용하면 안 된다.
    - contract에 새 build feature를 추가하면 STAGE0_SPEC_BY_OUTPUT에도 해당 output column spec이 있어야 한다.
    - build_action은 현재 build, encode만 허용된다.
    - fb_outputs는 후보 산출물이다. 학습 노트북은 사람이 검토해 승인한 ml_inputs를 읽는 구조다
4. 확인 필요 사항
    - `SOURCE_PARQUET_PATH`의 schema가 contract 작성 시점과 동일한지 실행 전 확인 필요.
    - full build 시간과 peak memory는 환경 의존이므로 sample run으로 확인 필요.
    - `fb_outputs`를 `ml_inputs`로 승격하는 승인 절차와 파일명 규칙은 팀 결정/확인 필요.

# 00_경로 및 환경설정

In [1]:
from __future__ import annotations

import importlib
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
from IPython.display import display

# ============================================================
# 0.1 Runtime Settings
# 실행 환경: ML 담당자 기준
# - Kernel          : Local WSL
# - Code repo       : local Git repo
# - Source parquet  : Google Drive mount
# - Input artifacts : local Git repo ml/ml-01/fb_inputs/
# - Output artifacts: local Git repo ml/ml-01/fb_outputs/
#
# 이 셀은 이후 모든 셀이 참조할 경로와 import 기준을 고정한다.
# 경로가 잘못되면 contract 로드, parquet read, output 저장 단계가 모두 다른 위치를 보게 된다.
# ============================================================
SEED = 42

# local repo는 코드/contract/output 후보를 관리하고, drive repo는 대용량 source parquet를 읽는 기준 경로다.
# 확인 필요: 다른 실행 환경에서는 mount 경로가 달라질 수 있으므로 이 두 경로를 먼저 확인한다.
LOCAL_REPO_ROOT = Path("/mnt/d/AML_git/Graph_AML").resolve()
DRIVE_REPO_ROOT = Path("/mnt/g/내 드라이브/Graph_AML").resolve()

# ML-01 하위에서 입력 contract, 출력 후보 산출물, feature build 모듈을 분리해 관리한다.
ML_01_BASE_DIR = LOCAL_REPO_ROOT / "ml" / "ml-01"

ML_01_FB_INPUTS_DIR = ML_01_BASE_DIR / "fb_inputs"
ML_01_FB_OUTPUTS_DIR = ML_01_BASE_DIR / "fb_outputs"
ML_01_MODULE_FB_DIR = ML_01_BASE_DIR / "00_feature_build"

# ============================================================
# 0.2 Path Validation
# ============================================================
# 잘못된 경로로 긴 feature build를 시작하지 않도록 실행 초기에 fail-fast 한다.
def require_dir(path: Path, name: str) -> None:
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{name} is not a directory: {path}")

for name, path in {
    "LOCAL_REPO_ROOT": LOCAL_REPO_ROOT,
    "DRIVE_REPO_ROOT": DRIVE_REPO_ROOT,
    "ML_01_BASE_DIR": ML_01_BASE_DIR,
    "ML_01_FB_INPUTS_DIR": ML_01_FB_INPUTS_DIR,
    "ML_01_FB_OUTPUTS_DIR": ML_01_FB_OUTPUTS_DIR,
    "ML_01_MODULE_FB_DIR": ML_01_MODULE_FB_DIR,
}.items():
    require_dir(path, name)

# ============================================================
# 0.3 Import Path Setup
# ============================================================
# 노트북이 로컬 모듈 파일을 import하도록 feature build 모듈 경로를 sys.path 앞쪽에 추가한다.
# 이 순서가 바뀌면 동일 이름의 다른 모듈을 import할 수 있으므로 변경 시 주의한다.
MODULE_DIR = str(ML_01_MODULE_FB_DIR)
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import ml_01_fb_build
import ml_01_fb_catalog
import ml_01_fb_contract
import ml_01_fb_encoding
import ml_01_fb_io
import ml_01_fb_operations
import ml_01_fb_rolling
import ml_01_fb_schema
import ml_01_fb_specs
import ml_01_fb_utils

# Notebook kernel이 이전 모듈 버전을 들고 있으면 rolling window 결과가 복제될 수 있다.
# 특히 ml_01_fb_rolling.py를 수정한 뒤 kernel을 유지하면 stale import가 남을 수 있다.
# 의존 모듈을 dependency -> caller 순서로 reload한 뒤 함수 symbol을 다시 import한다.
# reload 순서를 바꾸면 상위 모듈이 예전 하위 모듈 symbol을 참조할 수 있으므로 주의한다.
for module in (
    ml_01_fb_utils,
    ml_01_fb_specs,
    ml_01_fb_schema,
    ml_01_fb_io,
    ml_01_fb_catalog,
    ml_01_fb_rolling,
    ml_01_fb_operations,
    ml_01_fb_contract,
    ml_01_fb_encoding,
    ml_01_fb_build,
):
    importlib.reload(module)

from ml_01_fb_build import FeatureBuildConfig, build_features, validate_stage0_rolling_outputs
from ml_01_fb_contract import validate_fb_input_contract
from ml_01_fb_encoding import encode_split_frame, load_encoding_specs
from ml_01_fb_specs import ml01_stage0_feature_specs
from ml_01_fb_utils import set_seed

set_seed(SEED)

# 01_실행 설정

In [2]:
# ============================================================
# 1.1 Run identifiers
# ============================================================
# EXPERIMENT_ID와 RUN_ID는 입력 contract 파일명과 출력 후보 artifact 파일명을 동시에 결정한다.
# 같은 RUN_ID를 재사용하면 overwrite 정책에 따라 기존 산출물과 충돌할 수 있다.
EXPORT_EXPERIMENT_ID = "ml_01"
RUN_ID = "r01"
# ARTIFACT_PREFIX는 contract, manifest, parquet, feature_info 파일명을 연결하는 공통 prefix다.
ARTIFACT_PREFIX = f"{EXPORT_EXPERIMENT_ID}__{RUN_ID}"

# ============================================================
# 1.2 Source inputs
# ============================================================
# ML-01 Stage 0 feature를 만들 원본 parquet 경로다.
# 이 parquet의 기존 split 컬럼을 그대로 사용하므로, 여기서 데이터를 바꾸면 train/val/test 대상도 바뀐다.
SOURCE_PARQUET_PATH = DRIVE_REPO_ROOT / "data" / "processed" / "ml_features" / "ml_exp00.parquet"

# ============================================================
# 1.3 Output contract path
# ============================================================
# fb_inputs는 이전 contract 작성 단계의 승인 전 입력 계약을 읽는 위치다.
# fb_outputs는 이 노트북이 새로 만드는 후보 산출물 위치이며, 학습 입력으로 바로 확정하지 않는다.
FB_INPUT_DIR = ML_01_FB_INPUTS_DIR / RUN_ID
FB_OUTPUT_DIR = ML_01_FB_OUTPUTS_DIR / RUN_ID

FB_INPUT_CONTRACT_PATH = FB_INPUT_DIR / f"{ARTIFACT_PREFIX}_fb_input_feature_contract.csv"
FB_INPUT_CATEGORY_VALUES_PATH = FB_INPUT_DIR / f"{ARTIFACT_PREFIX}_category_values.json"

FB_OUTPUT_CONTRACT_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_fb_output_feature_contract.csv"
FB_OUTPUT_MANIFEST_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_encoding_manifest.json"
FB_OUTPUT_FEATURE_TYPES_PATH = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_feature_types.json"

# full artifact 생성 시 None을 사용한다. smoke/debug만 할 때 정수로 제한한다.
# SAMPLE_ROWS를 정수로 두면 parquet 앞부분 일부 row만 읽으므로 결과는 smoke/debug 전용이다.
# sample 결과로 모델 성능이나 feature 분포를 주장하면 안 된다.
SAMPLE_ROWS = None
# False이면 기존 fb_outputs 후보 파일을 보호한다. 재실행으로 덮어써야 할 때만 True로 바꾼다.
OVERWRITE_FB_OUTPUTS = False

# source parquet 컬럼명이 feature spec의 logical name과 다르면 여기에 명시한다.
# 예: spec은 amount를 기대하지만 source는 Amount Paid일 때 매핑한다.
# 잘못 지정하면 feature 계산 대상 컬럼 자체가 바뀌므로 schema 확인 후 수정한다.
COLUMN_MAP = {}


# 코드에 정의된 Stage 0 전체 후보 spec이다. contract가 요청한 output_col을 여기서 찾아 실제 계산 spec으로 연결한다.
ALL_STAGE0_FEATURE_SPECS = ml01_stage0_feature_specs(used_in_ml=True)
# output column name -> FeatureSpec lookup table. contract와 코드 spec 사이의 정합성 검증 기준이다.
STAGE0_SPEC_BY_OUTPUT = {spec.output_col: spec for spec in ALL_STAGE0_FEATURE_SPECS}

print("=" * 80)
print("SEED                  :", SEED)
print("-" * 80)
print("EXPORT_EXPERIMENT_ID  :", EXPORT_EXPERIMENT_ID)
print("RUN_ID                :", RUN_ID)
print("ARTIFACT_PREFIX       :", ARTIFACT_PREFIX)
print("-" * 80)
print("LOCAL_REPO_ROOT          :", LOCAL_REPO_ROOT)
print("DRIVE_REPO_ROOT          :", DRIVE_REPO_ROOT)
print("-" * 80)
print("SOURCE_PARQUET_PATH      :", SOURCE_PARQUET_PATH)
print("-" * 80)
print("ML_01_FB_INPUTS_DIR      :", ML_01_FB_INPUTS_DIR)
print("FB_INPUT_DIR             :", FB_INPUT_DIR)
print("FB_INPUT_CONTRACT_PATH   :", FB_INPUT_CONTRACT_PATH)
print("FB_INPUT_CATEGORY_VALUES :", FB_INPUT_CATEGORY_VALUES_PATH)
print("ML_01_MODULE_FB_DIR      :", ML_01_MODULE_FB_DIR)
print("-" * 80)
print("ML_01_FB_OUTPUTS_DIR     :", ML_01_FB_OUTPUTS_DIR)
print("FB_OUTPUT_DIR            :", FB_OUTPUT_DIR)
print("FB_OUTPUT_CONTRACT_PATH  :", FB_OUTPUT_CONTRACT_PATH)
print("FB_OUTPUT_MANIFEST_PATH  :", FB_OUTPUT_MANIFEST_PATH)
print("=" * 80)

SEED                  : 42
--------------------------------------------------------------------------------
EXPORT_EXPERIMENT_ID  : ml_01
RUN_ID                : r01
ARTIFACT_PREFIX       : ml_01__r01
--------------------------------------------------------------------------------
LOCAL_REPO_ROOT          : /mnt/d/AML_git/Graph_AML
DRIVE_REPO_ROOT          : /mnt/g/내 드라이브/Graph_AML
--------------------------------------------------------------------------------
SOURCE_PARQUET_PATH      : /mnt/g/내 드라이브/Graph_AML/data/processed/ml_features/ml_exp00.parquet
--------------------------------------------------------------------------------
ML_01_FB_INPUTS_DIR      : /mnt/d/AML_git/Graph_AML/ml/ml-01/fb_inputs
FB_INPUT_DIR             : /mnt/d/AML_git/Graph_AML/ml/ml-01/fb_inputs/r01
FB_INPUT_CONTRACT_PATH   : /mnt/d/AML_git/Graph_AML/ml/ml-01/fb_inputs/r01/ml_01__r01_fb_input_feature_contract.csv
FB_INPUT_CATEGORY_VALUES : /mnt/d/AML_git/Graph_AML/ml/ml-01/fb_inputs/r01/ml_01__r01_category_v

# 02_contract 로드 및 build 대상 확정

In [3]:
# ============================================================
# 02_contract 로드 및 build 대상 확정
# ============================================================
# 이 셀의 역할:
# 1. 이전 단계에서 작성된 fb_input_feature_contract.csv가 현재 실행 설정과 맞는지 검증한다.
# 2. contract를 DataFrame으로 읽어 build 대상 feature와 encoding 대상 feature를 분리한다.
# 3. contract가 요청한 build feature가 실제 Stage 0 feature spec에 존재하는지 확인한다.
# 4. 이후 build_features()와 encode_split_frame()에 넘길 BUILD_FEATURE_SPECS, ENCODING_SPECS를 확정한다.
#
# 중요:
# - 이 셀은 feature를 실제로 만들지는 않는다.
# - 이 셀은 "무엇을 만들 것인가"를 contract 기준으로 확정하는 단계다.
# - 여기서 잘못 분류되면 다음 셀의 build/export 결과 전체가 달라진다.
# ============================================================

# contract 파일이 현재 source parquet, artifact prefix와 맞는지 먼저 검증한다.
# 이 검증이 실패하면 contract가 다른 데이터/실행 ID 기준으로 만들어졌을 가능성이 있다.
validation = validate_fb_input_contract(
    FB_INPUT_CONTRACT_PATH,
    source_data_path=SOURCE_PARQUET_PATH,
    artifact_prefix=ARTIFACT_PREFIX,
)

# feature contract CSV를 DataFrame으로 로드한다.
# 이 DataFrame은 이후 build/encode 대상 분리, used_in_ml 전달, output contract metadata 보존에 사용된다.
# used_in_ml, build_in_fb, materialized는 TRUE/FALSE 같은 문자열 플래그로 비교하므로 string dtype을 명시한다.
contract_df = pd.read_csv(
    FB_INPUT_CONTRACT_PATH,
    encoding="utf-8-sig",
    dtype={"used_in_ml": "string", "build_in_fb": "string", "materialized": "string"},
)

# ML 입력으로 사용하는 feature 중 XGBoost categorical type("c")로 표시된 컬럼명을 추출한다.
# 여기서는 집계/검증용 count만 만들고, 실제 encoding 대상 생성은 ENCODING_SPECS가 담당한다.
categorical_columns = contract_df.loc[
    (contract_df["used_in_ml"].astype(str).str.strip() == "TRUE")
    & (contract_df["xgb_feature_type"].astype(str).str.strip() == "c"),
    "column_name",
].astype(str).str.strip().tolist()

# categorical feature 중 build_action이 "encode"가 아닌 컬럼을 따로 분리한다.
# 수동 category 처리 대상이 있으면 category_values.json 없이 실행하면 mapping 기준을 알 수 없다.
manual_categorical_columns = contract_df.loc[
    (contract_df["used_in_ml"].astype(str).str.strip() == "TRUE")
    & (contract_df["xgb_feature_type"].astype(str).str.strip() == "c")
    & (contract_df["build_action"].astype(str).str.strip() != "encode"),
    "column_name",
].astype(str).str.strip().tolist()

# 수동 categorical feature가 있으면 현재 실행에서 사용할 category values 파일이 반드시 필요하다.
if manual_categorical_columns and not FB_INPUT_CATEGORY_VALUES_PATH.is_file():
    raise FileNotFoundError(
        "manual categorical columns require current-run category values. "
        f"missing={FB_INPUT_CATEGORY_VALUES_PATH}"
    )

# contract의 build_action, build_in_fb 컬럼을 정규화해 이후 필터 조건에 재사용한다.
# 문자열 공백 차이로 build/encode 분류가 흔들리지 않도록 한 번만 정규화한다.
build_action = contract_df["build_action"].astype(str).str.strip()
build_in_fb = contract_df["build_in_fb"].astype(str).str.strip()

# feature builder가 직접 생성해야 하는 feature 행만 추출한다.
# build_in_fb=TRUE이고 build_action=build인 행은 STAGE0_SPEC_BY_OUTPUT에 정의된 build spec과 매칭된다.
build_rows = contract_df[(build_in_fb == "TRUE") & (build_action == "build")].copy()

# feature builder가 encoding 방식으로 생성해야 하는 feature 행만 추출한다.
# 실제 encoding spec 생성은 아래 load_encoding_specs()에서 처리된다.
encode_rows = contract_df[(build_in_fb == "TRUE") & (build_action == "encode")].copy()

# build_in_fb=TRUE인데 build_action이 허용값이 아닌 행을 잡아낸다. 허용되는 action은 "build", "encode"
unsupported_build_rows = contract_df[(build_in_fb == "TRUE") & ~build_action.isin(["build", "encode"])].copy()

# contract에 잘못된 build_action이 있으면 feature 생성 전에 중단한다.
if not unsupported_build_rows.empty:
    raise ValueError(
        "build_in_fb=TRUE rows must use build_action in {'build', 'encode'}. "
        f"bad_rows={unsupported_build_rows[['column_name', 'build_action']].head(30).to_dict('records')}"
    )

# build 대상 feature의 최종 출력 컬럼명을 리스트로 만든다.
build_output_cols = build_rows["column_name"].astype(str).str.strip().tolist()

# build 대상 feature가 하나도 없으면 이후 feature build 단계가 의미 없으므로 실패시킨다.
if not build_output_cols:
    raise ValueError("No build_in_fb=TRUE rows found in final contract. Nothing to build.")

# contract가 요청한 build feature가 Stage 0 spec에 모두 정의되어 있는지 확인한다.
missing_specs = sorted(set(build_output_cols) - set(STAGE0_SPEC_BY_OUTPUT))

# 누락된 spec이 있으면 feature build를 진행하지 않는다.
if missing_specs:
    raise ValueError(f"contract requests build columns that are not in Stage 0 specs: {missing_specs[:30]}")

# 실제 feature build에 사용할 spec 튜플을 만든다.
# STAGE0_SPEC_BY_OUTPUT에서 기본 spec을 가져오고, used_in_ml 값만 contract 기준으로 덮어쓴다.
# 즉, 계산 방식은 코드 spec이 결정하고 ML 사용 여부는 contract가 결정한다.
BUILD_FEATURE_SPECS = tuple(
    replace(
        STAGE0_SPEC_BY_OUTPUT[str(row["column_name"]).strip()],
        used_in_ml=str(row["used_in_ml"]).strip() == "TRUE",
    )
    for _, row in build_rows.iterrows()
)

# encoding 대상 feature spec을 contract 파일에서 로드한다.
# encode_rows는 count 출력에만 쓰이고, 실제 ENCODING_SPECS 구성은 load_encoding_specs 내부 로직에 의존한다.
# 확인 필요: encoding 정책을 바꿀 때는 ml_01_fb_encoding.py와 contract 컬럼 정의를 함께 확인한다.
ENCODING_SPECS = load_encoding_specs(FB_INPUT_CONTRACT_PATH)

# contract 검증 결과와 feature 분류 결과를 출력한다.
print("contract_total_rows", validation.total_rows)
print("contract_selected_count", validation.selected_count)
print("categorical_feature_count", len(categorical_columns))
print("manual_categorical_feature_count", len(manual_categorical_columns))
print("build_feature_count", len(BUILD_FEATURE_SPECS))
print("encode_feature_count", len(encode_rows))

# build 대상 contract 행을 화면에 표시한다.
display(build_rows)

contract_total_rows 104
contract_selected_count 77
categorical_feature_count 10
manual_categorical_feature_count 0
build_feature_count 62
encode_feature_count 10


,contract_version,artifact_prefix,column_name,used_in_ml,source_column,column_origin,encoding,build_action,build_in_fb,xgb_feature_type,...,source_data_path,feature_spec_name,encoding_params,materialized,observed_dtype,missing_count,missing_rate,unknown_category_count,fit_split,note
42,1,ml_01__r01,timehist__sender__out__tx_count__count__w1h,TRUE,timehist__sender__out__tx_count__count__w1h,current_build,passthrough,build,TRUE,q,...,/mnt/g/내 드라이브/Graph_AML/data/processed/ml_feat...,rolling_agg,"{""input_cols"": {""entity_col"": ""sender_account_...",FALSE,NaN,NaN,NaN,NaN,NaN,to be built by 01_ml_01_fb_build_features.ipynb
43,1,ml_01__r01,timehist__sender__out__amount__sum__w1h,TRUE,timehist__sender__out__amount__sum__w1h,current_build,passthrough,build,TRUE,q,...,/mnt/g/내 드라이브/Graph_AML/data/processed/ml_feat...,rolling_agg,"{""input_cols"": {""entity_col"": ""sender_account_...",FALSE,NaN,NaN,NaN,NaN,NaN,to be built by 01_ml_01_fb_build_features.ipynb
44,1,ml_01__r01,timehist__sender__out__amount__mean__w1h,TRUE,timehist__sender__out__amount__mean__w1h,current_build,passthrough,build,TRUE,q,...,/mnt/g/내 드라이브/Graph_AML/data/processed/ml_feat...,rolling_agg,"{""input_cols"": {""entity_col"": ""sender_account_...",FALSE,NaN,NaN,NaN,NaN,NaN,to be built by 01_ml_01_fb_build_features.ipynb
45,1,ml_01__r01,timehist__sender__out__amount__std__w1h,TRUE,timehist__sender__out__amount__std__w1h,current_build,passthrough,build,TRUE,q,...,/mnt/g/내 드라이브/Graph_AML/data/processed/ml_feat...,rolling_agg,"{""input_cols"": {""entity_col"": ""sender_account_...",FALSE,NaN,NaN,NaN,NaN,NaN,to be built by 01_ml_01_fb_build_features.ipynb
46,1,ml_01__r01,timehist__sender__out__amount__max__w1h,TRUE,timehist__sender__out__amount__max__w1h,current_build,passthrough,build,TRUE,q,...,/mnt/g/내 드라이브/Graph_AML/data/processed/ml_feat...,rolling_agg,"{""input_cols"": {""entity_col"": ""sender_account_...",FALSE,NaN,NaN,NaN,NaN,NaN,to be built by 01_ml_01_fb_build_features.ipynb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,1,ml_01__r01,timehist__receiver__in__amount__cur_vs_mean_ra...,TRUE,timehist__receiver__in__amount__cur_vs_mean_ra...,current_build,passthrough,build,TRUE,q,...,/mnt/g/내 드라이브/Graph_AML/data/processed/ml_feat...,cur_vs_mean_ratio,"{""input_cols"": {""entity_col"": ""receiver_accoun...",FALSE,NaN,NaN,NaN,NaN,NaN,to be built by 01_ml_01_fb_build_features.ipynb
100,1,ml_01__r01,timehist__receiver__in__amount__cur_vs_mean_ra...,TRUE,timehist__receiver__in__amount__cur_vs_mean_ra...,current_build,passthrough,build,TRUE,q,...,/mnt/g/내 드라이브/Graph_AML/data/processed/ml_feat...,cur_vs_mean_ratio,"{""input_cols"": {""entity_col"": ""receiver_accoun...",FALSE,NaN,NaN,NaN,NaN,NaN,to be built by 01_ml_01_fb_build_features.ipynb
101,1,ml_01__r01,timehist__receiver__in__amount__cur_vs_mean_ra...,TRUE,timehist__receiver__in__amount__cur_vs_mean_ra...,current_build,passthrough,build,TRUE,q,...,/mnt/g/내 드라이브/Graph_AML/data/processed/ml_feat...,cur_vs_mean_ratio,"{""input_cols"": {""entity_col"": ""receiver_accoun...",FALSE,NaN,NaN,NaN,NaN,NaN,to be built by 01_ml_01_fb_build_features.ipynb
102,1,ml_01__r01,timehist__sender__all__tx_count__cum__whist,TRUE,timehist__sender__all__tx_count__cum__whist,current_build,passthrough,build,TRUE,q,...,/mnt/g/내 드라이브/Graph_AML/data/processed/ml_feat...,cumulative_count,"{""input_cols"": {""entity_col"": ""sender_account_...",FALSE,NaN,NaN,NaN,NaN,NaN,to be built by 01_ml_01_fb_build_features.ipynb


# 03_feature build 실행

In [4]:
# ============================================================
# 03_feature build 실행
# ============================================================
# 이 셀의 역할:
# 1. SOURCE_PARQUET_PATH 원본 parquet에서 ML-01 Stage 0 feature를 계산한다.
# 2. contract에서 확정한 BUILD_FEATURE_SPECS만 실제 build 대상으로 사용한다.
# 3. build된 전체 feature frame에 ENCODING_SPECS를 적용한다.
# 4. fb_outputs에 Xy_all, Xy_train, Xy_val, Xy_test, output contract, manifest를 저장한다.
# 5. feature_info.csv를 별도로 저장해 어떤 feature가 어떻게 생성됐는지 추적 가능하게 한다.
#
# 중요:
# - 이 노트북은 ml_inputs 승인본을 읽지 않고 fb_outputs 후보 산출물을 만든다.
# - guard 검증 전에는 후보 parquet/contract를 저장하지 않는 흐름을 유지해야 한다.
# - full run은 parquet 전체 로드와 rolling feature 계산을 수행하므로 sample smoke 후 실행하는 것이 안전하다.
# ============================================================

# FeatureBuildConfig는 build_features() 실행 조건을 하나로 묶는 설정 객체다.
# 여기 들어가는 값이 "어떤 데이터에서", "어떤 feature를", "어떤 실행 ID로" 만들지를 결정한다.
config = FeatureBuildConfig(
    input_path=SOURCE_PARQUET_PATH,  # feature를 만들 원본 parquet 경로다.

    # build_features() 단계에서는 파일을 직접 저장하지 않고 메모리 결과만 반환하게 한다.
    # 실제 parquet 저장은 아래 encode_split_frame()에서 FB_OUTPUT_DIR 기준으로 수행한다.
    output_dir=None,

    experiment_id=ARTIFACT_PREFIX,
    run_name="ml01_stage0_time_history",

    # 실제로 계산할 feature spec 목록이다.
    # 앞 셀에서 contract의 build_in_fb=TRUE, build_action=build 조건으로 확정된 feature만 들어간다.
    feature_specs=BUILD_FEATURE_SPECS,

    # source parquet의 실제 컬럼명과 feature spec이 기대하는 logical column명이 다를 때 사용하는 매핑이다.
    # COLUMN_MAP이 빈 dict이면 None으로 넘겨 기본 컬럼명을 사용한다.
    column_map=COLUMN_MAP or None,

    # None이면 full parquet를 읽고, 정수이면 앞쪽 일부 row만 읽어 smoke build를 수행한다.
    sample_rows=SAMPLE_ROWS,

    # output_dir=None이라 build_features 단계 자체는 파일을 저장하지 않지만, config 계약상 overwrite 값을 함께 넘긴다.
    overwrite=OVERWRITE_FB_OUTPUTS,
)


# 원본 parquet를 읽고 BUILD_FEATURE_SPECS에 정의된 feature를 계산한다.
# 반환되는 build_result의 핵심:
# - feature_frame: 원본 메타 컬럼 + 기존 feature + 새로 build된 feature가 들어 있는 전체 DataFrame
# - feature_info: 생성된 feature별 metadata/설명/스펙 정보
# - row_counts: build 전후 row 수 확인용 정보
#
# 주의:
# - 이 단계에서는 아직 categorical encoding 결과 parquet가 저장되지 않는다.
# - SOURCE_PARQUET_PATH의 기존 split 컬럼은 feature_frame 안에 유지되어 이후 train/val/test 분리에 사용된다.
# - build_features()는 이 경로에서 split을 새로 만들지 않는다.
build_result = build_features(config)

# rolling semantic guard: 이 검증을 통과해야만 fb_outputs 저장 단계로 진행한다.
# 특히 w1h/w6h/w1d/w3d/w7d 결과가 stale import로 같은 값으로 복제되는 문제를 차단한다.
# 또한 count/sum window 단조성, duplicate timestamp history 누수 여부를 확인한다.
# 이 줄보다 아래에 저장 로직이 있어야 guard 실패 시 artifact가 남지 않는다.
stage0_rolling_validation = validate_stage0_rolling_outputs(build_result.feature_frame)
print("stage0 rolling validation", stage0_rolling_validation)


# build_result.feature_frame에 ENCODING_SPECS를 적용하고 fb_outputs 후보 산출물을 저장한다.
# 이 함수 호출부터는 실제 파일 write가 발생하므로 output_dir, artifact_prefix, overwrite 값을 특히 확인한다.
# 이 함수가 담당하는 일:
# - contract 기준으로 passthrough / label_code / xgb_native encoding 적용
# - train split 기준으로 category mapping fit
# - validation/test의 새로운 category는 train 기준 mapping 밖의 값으로 처리
# - Xy_all, Xy_train, Xy_val, Xy_test parquet 저장
# - fb_output_feature_contract.csv 저장
# - encoding_manifest.json 저장
# - feature_types.json, category mapping, unknown category summary, split summary 저장

encoding_result = encode_split_frame(    
    build_result.feature_frame,           # encoding과 split export의 입력이 되는 전체 feature frame이다.    
    ENCODING_SPECS,                       # contract에서 로드한 encoding 대상 spec 목록이다.    
    output_dir=FB_OUTPUT_DIR,   # 후보 parquet/contract/manifest가 저장될 run별 출력 디렉터리다.
    artifact_prefix=ARTIFACT_PREFIX,
    overwrite=OVERWRITE_FB_OUTPUTS,
    input_label=str(SOURCE_PARQUET_PATH), # manifest에 기록될 입력 데이터 경로 문자열이다.

    # output contract 작성 시 원본 contract 정보를 함께 넘긴다.
    # 이 값이 있어야 used_in_ml, feature group, review 관련 contract metadata를 유지할 수 있다.
    contract_table=contract_df,
)

# feature_info.csv를 추가 저장하기 위해 출력 디렉터리를 보장한다.
# encode_split_frame()도 output_dir를 쓰지만, feature_info 저장 전 한 번 더 안전하게 확인한다.
FB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# build 단계의 feature metadata 저장 경로다.
# output contract가 "최종 materialized feature 계약"에 가깝다면,
# feature_info는 "build_features가 어떤 feature를 어떤 spec으로 만들었는지"를 추적하는 보조 산출물이다.
feature_info_path = FB_OUTPUT_DIR / f"{ARTIFACT_PREFIX}_feature_info.csv"

# overwrite=False일 때 기존 feature_info.csv를 덮어쓰지 않는다.
# 이 보호 장치가 없으면 같은 RUN_ID 재실행으로 이전 feature metadata가 조용히 교체될 수 있다.
if feature_info_path.exists() and not OVERWRITE_FB_OUTPUTS:
    raise FileExistsError(f"output already exists: {feature_info_path}")


# feature build metadata를 CSV로 저장한다.
# feature_info는 학습 입력 컬럼 자체가 아니라 생성 과정과 품질 통계를 추적하기 위한 관리 파일이다.
build_result.feature_info.to_csv(feature_info_path, index=False, encoding="utf-8-sig")


# 실행 결과 요약 로그다.
print("feature build completed")
print("row_counts", build_result.row_counts)  # build_features() 기준 row count다.

# encode_split_frame() 기준 row count다.
# all = train + val + test 관계가 맞아야 한다.
print("encoding row_counts", encoding_result.row_counts)

# split 전후 schema/row count 점검용 전체 parquet 경로다.
# 학습용 입력으로 쓰지 않고, 전체 산출물 관리와 검토에 사용한다.
print("xy_all_path", encoding_result.output_paths.all_path)

# output feature contract 경로다.
# fb_outputs 후보 contract이며, 검토 후 ml_inputs의 approve contract로 배치해야 학습 입력 기준이 된다.
print("fb_output contract ready", encoding_result.output_paths.feature_contract_path)

# encoding manifest 경로다.
# categorical feature의 category mapping, feature type, 산출물 경로를 추적하는 핵심 metadata다.
print("encoding_manifest_path", FB_OUTPUT_MANIFEST_PATH)

stage0 rolling validation {'rows': 5077237, 'monotonic_checks': 16, 'diversity_checks': 6, 'duplicate_timestamp_checks': 12, 'validated_columns': ['timehist__receiver__in__amount__cur_vs_mean_ratio__w1d', 'timehist__receiver__in__amount__cur_vs_mean_ratio__w7d', 'timehist__receiver__in__amount__sum__w1d', 'timehist__receiver__in__amount__sum__w1h', 'timehist__receiver__in__amount__sum__w3d', 'timehist__receiver__in__amount__sum__w6h', 'timehist__receiver__in__amount__sum__w7d', 'timehist__receiver__in__tx_count__count__w1d', 'timehist__receiver__in__tx_count__count__w1h', 'timehist__receiver__in__tx_count__count__w3d', 'timehist__receiver__in__tx_count__count__w6h', 'timehist__receiver__in__tx_count__count__w7d', 'timehist__sender__out__amount__cur_vs_mean_ratio__w1d', 'timehist__sender__out__amount__cur_vs_mean_ratio__w7d', 'timehist__sender__out__amount__sum__w1d', 'timehist__sender__out__amount__sum__w1h', 'timehist__sender__out__amount__sum__w3d', 'timehist__sender__out__amount__su

## ML 실행 연결

feature build가 만든 `fb_outputs` 후보 산출물을 사람이 검토한 뒤, 승인본을 `ml_inputs`에 아래 패턴으로 배치한다. 학습 노트북은 `ml_inputs`의 승인본만 읽는다.

`*_Xy_all.parquet`는 train/val/test를 섞어 학습하기 위한 파일이 아니라, split 전후 row count와 schema를 점검하기 위한 관리 산출물이다.

```text
ml/ml-01/ml_inputs/r00/ml_01__r00_Xy_all.parquet
ml/ml-01/ml_inputs/r00/ml_01__r00_Xy_train.parquet
ml/ml-01/ml_inputs/r00/ml_01__r00_Xy_val.parquet
ml/ml-01/ml_inputs/r00/ml_01__r00_Xy_test.parquet
ml/ml-01/ml_inputs/r00/ml_01__r00_feature_contract_approve.csv
ml/ml-01/ml_inputs/r00/ml_01__r00_encoding_manifest.json
```